# 07 · k-Nearest Neighbors (k-NN)
> **Part:** Part 1 · Classical supervised learning  
> **Family:** `Supervised · Classification / Regression (instance-based, non-parametric)`

## Definition
k-NN does no training at all. To classify a new point it finds the k most similar stored examples (nearest by distance)
and takes a majority vote (classification) or an average (regression). The "model" *is* the training data.

## Intuition
"Tell me who your neighbours are and I'll tell you who you are." A small k follows every quirk of the data (low bias,
high variance); a large k blurs everything toward the global majority (high bias, low variance).


## The Math

```text
d(x, xᵢ) = ||x − xᵢ||₂      (or Manhattan, cosine, ...)
    ŷ(x) = argmax_c  Σ_{i ∈ N_k(x)} 1[yᵢ = c]            regression: ŷ = mean of the k neighbours' y
    Weighted variant: each neighbour votes with weight 1/d.
```


In [ ]:
# Setup, Imports & Styling
%matplotlib inline
import os
import pathlib
import sys
from IPython.display import Image, display

# Ensure parent folder is in path to import mlatlas
parent_dir = pathlib.Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlatlas import data, viz, cli

# Ensure interactive inline mode and configure Seaborn theme
plt.ion()
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})


### Step 1: Spatial Class Distribution Visualization
We visualize interlocking two moons clusters to explore instance-based non-linear boundaries.


In [ ]:
X, y = data.moons(n=300, noise=0.25, seed=0)

plt.figure(figsize=(8, 5))
sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, palette=["#4c78a8", "#f58518"], s=55, edgecolor="k")
plt.title("Two Moons Dataset: Non-Linear Decision Geometry", fontsize=12, fontweight="bold")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.tight_layout()
plt.show()


### Step 2: From-Scratch Algorithm Implementation
Below is the complete implementation of the model components in pure NumPy.


In [ ]:
# Helper Classes & Mathematical Logic
def votes(order, ytr, k, K=2):
    nn = ytr[order[:, :k]]; cnt = np.stack([(nn == c).sum(1) for c in range(K)], 1)
    return cnt.argmax(1), cnt[:, 1] / k


In [ ]:
# Main Algorithm & Training Routine
def main(gif, quick):
    Xtr, ytr = data.moons(200, .35, seed=1); Xte, yte = data.moons(200, .35, seed=2)
    xx, yy, G = viz.grid(Xtr, n=70 if quick else 100)
    sq = lambda A, B: ((A[:, None, :] - B[None]) ** 2).sum(-1)
    og, ot, otr = np.argsort(sq(G, Xtr), 1), np.argsort(sq(Xte, Xtr), 1), np.argsort(sq(Xtr, Xtr), 1)
    ks = [1, 3, 5, 9, 15, 25, 41, 61]; allk = list(range(1, 62, 2))
    acc_te = [(votes(ot, ytr, k)[0] == yte).mean() for k in allk]; acc_tr = [(votes(otr, ytr, k)[0] == ytr).mean() for k in allk]
    print(f"  best k = {allk[int(np.argmax(acc_te))]} (test accuracy {max(acc_te):.2f})")
    fig, (a1, a2) = viz.subplots(1, 2, w=11, h=4.4)

    def update(k):
        a1.clear(); a2.clear(); _, P = votes(og, ytr, k); viz.show_map(a1, xx, yy, P, prob=True)
        a1.contour(xx, yy, P.reshape(xx.shape), levels=[.5], colors="k", linewidths=1.5); viz.scatter(a1, Xtr, ytr)
        a1.set(title=f"k = {k}: decision regions", xlim=(xx.min(), xx.max()), ylim=(yy.min(), yy.max()))
        a2.plot(allk, acc_tr, color=viz.PALETTE[2], label="train"); a2.plot(allk, acc_te, color=viz.PALETTE[3], label="test")
        a2.axvline(k, color="k", ls="--"); a2.set(title="Accuracy vs k (small k overfits, large k underfits)", xlabel="k", ylim=(.6, 1.02)); a2.legend()
    viz.animate(fig, update, ks, gif, fps=2)
    # ---- animation 2: one query, five neighbours
    k = 5; T = 18 if quick else 45; path = np.c_[np.linspace(-1.2, 2.2, T), .9 * np.sin(np.linspace(0, 2 * np.pi, T)) * .6 + .25]
    fig, ax = viz.subplots(1, 1, w=6.4, h=4.6)

    def update2(t):
        ax.clear(); q = path[t]; d = ((Xtr - q) ** 2).sum(1); nn = np.argsort(d)[:k]; lab = np.bincount(ytr[nn], minlength=2).argmax()
        viz.scatter(ax, Xtr, ytr, s=14)
        for i in nn: ax.plot([q[0], Xtr[i, 0]], [q[1], Xtr[i, 1]], color="k", lw=1.2, alpha=.7)
        ax.scatter(*q, marker="*", s=260, color=viz.PALETTE[lab], edgecolor="k", linewidth=1.5, zorder=6)
        ax.set(title=f"Query's {k} neighbours vote: {np.bincount(ytr[nn], minlength=2)}  ->  class {lab}", xlim=(-1.9, 2.9), ylim=(-1.4, 1.9))
    viz.animate(fig, update2, range(T), gif.replace(".gif", "_query.gif"), fps=8)


### Step 3: Live Decision Boundary Surfaces across k
We compare decision boundaries for $k=1$ (flexible, sensitive to noise), $k=15$ (balanced), and $k=51$ (smooth, oversmoothed).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
ks = [1, 15, 51]
titles = ["k = 1: High Variance (Memorization)", "k = 15: Balanced Boundary", "k = 51: High Bias (Oversmoothed)"]

x0_min, x0_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
x1_min, x1_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
gx, gy = np.meshgrid(np.linspace(x0_min, x0_max, 100), np.linspace(x1_min, x1_max, 100))
grid_flat = np.c_[gx.ravel(), gy.ravel()]

for ax, k, title in zip(axes, ks, titles):
    dists = np.linalg.norm(grid_flat[:, None, :] - X[None, :, :], axis=2)
    nn_idx = np.argpartition(dists, k, axis=1)[:, :k]
    nn_labels = y[nn_idx]
    preds = (nn_labels.mean(axis=1) >= 0.5).astype(int).reshape(gx.shape)
    
    ax.contourf(gx, gy, preds, levels=[-0.5, 0.5, 1.5], colors=["#4c78a8", "#f58518"], alpha=0.35)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=25, edgecolor="k")
    ax.set_title(title, fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()


### Step 4: Industry Benchmark (Comparison with scikit-learn)
We compare against `sklearn.neighbors.KNeighborsClassifier`.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

sk_knn = KNeighborsClassifier(n_neighbors=15).fit(X, y)
print("scikit-learn k-NN (k=15) Accuracy:", sk_knn.score(X, y).round(4))


### Step 5: Visualizing Learning Dynamics

1. `07_k_nearest_neighbors.gif`: the decision regions for k = 1 … 61 on noisy moons, plus train/test accuracy. k=1 is
   a jagged island map with 100 % training accuracy; huge k is too smooth.
2. `07_k_nearest_neighbors_query.gif`: a query point glides through the data; lines connect it to its 5 neighbours who vote.


In [ ]:
# Display the learning animation inline
media_file = f"../outputs/07_k_nearest_neighbors.gif"
if os.path.exists(media_file):
    display(Image(filename=media_file))
elif os.path.exists(f"outputs/07_k_nearest_neighbors.gif"):
    display(Image(filename=f"outputs/07_k_nearest_neighbors.gif"))
media_file = f"../outputs/07_k_nearest_neighbors_query.gif"
if os.path.exists(media_file):
    display(Image(filename=media_file))
elif os.path.exists(f"outputs/07_k_nearest_neighbors_query.gif"):
    display(Image(filename=f"outputs/07_k_nearest_neighbors_query.gif"))


## Practical tips

- **Scale features** (distance is dominated by large-range features). Beware the curse of dimensionality.
- Prediction cost grows with the dataset (use KD-trees / approximate NN such as FAISS or HNSW).
- k chosen by cross-validation; odd k avoids ties in binary problems.


## 🧠 Check Your Understanding

<details>
<summary><b>1. Why is k-NN described as a 'lazy learner'?</b></summary>
Because it does no explicit computation during training ($O(1)$ fit time); it simply stores the training instances. All computation is deferred until inference time, when distances to all training points must be computed ($O(n \cdot d)$).
</details>

<details>
<summary><b>2. How does the curse of dimensionality impair k-NN in high dimensions?</b></summary>
In high-dimensional feature spaces, the volume of space grows exponentially with dimension. Data points become sparse, and the distance to the nearest neighbor approaches the distance to the average neighbor, making proximity metrics uninformative.
</details>


## 🧪 Guided Experiments

- Multiply one feature by 100 before computing distances: watch the model ignore the other feature.
- Implement distance-weighted voting and compare accuracy at k = 15.
